In [ ]:
%pip install -U "reality_stone[full]" transformers


In [ ]:
import os
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from reality_stone.models.transformer_converter import (
    TransformerToRSULFConverter,
    cache_rsulf_hidden_states,
    finetune_lm_head_from_cache,
)


class TextLineDataset(Dataset):
    def __init__(self, file_path, tokenizer, max_length=128):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.lines = []
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if len(line) > 10:
                    self.lines.append(line)

    def __len__(self):
        return len(self.lines)

    def __getitem__(self, idx):
        text = self.lines[idx]
        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }


def run_rsulf_colab(
    model_name="mistralai/Mistral-7B-v0.1",
    save_dir="/content/mistral-7b-rsulf",
    device=None,
    cache_dir="/content/hf-cache",
    offline=False,
    folding_ratio=0.5,
    metric_strategy="diagonal",
    lr=0.02,
    alpha=0.04,
    beta=0.01,
    gamma=0.98,
    graph_window=8,
    graph_decay=0.9,
    fast_mode=True,
    max_layers=4,
    skip_tests=True,
    test_generation=False,
    test_prompt="Reality Stone은",
    max_length=50,
    finetune_text=None,
    finetune_epochs=1,
    finetune_batch_size=4,
    finetune_lr=1e-4,
    finetune_max_samples=5000,
    finetune_cache_file=None,
    quiet=False,
):
    os.environ.setdefault("HF_HOME", cache_dir)
    if offline:
        os.environ["TRANSFORMERS_OFFLINE"] = "1"
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cuda" and not torch.cuda.is_available():
        device = "cpu"
    if not quiet:
        print(f"device = {device}")
        print(f"model_name = {model_name}")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto" if device == "cuda" else "cpu",
        cache_dir=cache_dir,
        local_files_only=offline,
    )
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        cache_dir=cache_dir,
        local_files_only=offline,
    )
    if not quiet:
        num_params = sum(p.numel() for p in model.parameters())
        print(f"Transformer params: {num_params:,}")
    converter_config = {
        "metric_strategy": metric_strategy,
        "lr": lr,
        "alpha": alpha,
        "beta": beta,
        "gamma": gamma,
        "folding_ratio": folding_ratio,
        "graph_window_size": graph_window,
        "graph_directed": True,
        "graph_decay": graph_decay,
        "run_consistency_tests": not skip_tests,
        "consistency_tolerance": 1e-2,
        "verbose": not quiet,
        "fast_mode": fast_mode,
        "max_layers": max_layers,
    }
    converter = TransformerToRSULFConverter(config=converter_config)
    rs_model = converter.convert_model(model, device=device)
    if not quiet and hasattr(rs_model, "param_count"):
        stats = rs_model.param_count()
        rs_params = int(stats.get("compressed", 0))
        print("\nRS-ULF 변환 완료")
        print(f"RS-ULF params: {rs_params:,}")
        if folding_ratio is not None and num_params > 0:
            reduction = (1 - rs_params / num_params) * 100
            print(f"param reduction: {reduction:.2f}%")
    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)
    import json
    meta = dict(converter_config)
    meta["original_model_name"] = model_name
    with open(save_path / "converter_config.json", "w") as f:
        json.dump(meta, f, indent=2)
    if not quiet:
        print(f"config saved to: {save_path / 'converter_config.json'}")
    if test_generation:
        with torch.no_grad():
            inputs = tokenizer(test_prompt, return_tensors="pt").to(device)
            input_ids = inputs["input_ids"]
            embeds = model.model.embed_tokens(input_ids)
            seq_len = embeds.size(1)
            rs_model.update_graph_laplacians(seq_len, device=device)
            output, v_list = rs_model(embeds)
        if not quiet:
            print(f"input shape: {embeds.shape}")
            print(f"output shape: {output.shape}")
            print(f"memory states: {len(v_list)}")
    if finetune_text is not None:
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        dataset = TextLineDataset(finetune_text, tokenizer, max_length=max_length)
        loader = DataLoader(dataset, batch_size=finetune_batch_size, shuffle=True)
        if finetune_cache_file is None:
            cache_path = str(save_path / "rsulf_hidden_cache.pt")
        else:
            cache_path = finetune_cache_file
        if not os.path.exists(cache_path):
            if not quiet:
                print("RS-ULF hidden cache 생성 중...")
            cache_rsulf_hidden_states(
                model,
                rs_model,
                loader,
                cache_path=cache_path,
                device=device,
                max_samples=finetune_max_samples,
            )
        if not quiet:
            print("lm_head 파인튜닝 시작...")
        finetune_lm_head_from_cache(
            model,
            cache_path=cache_path,
            num_steps=finetune_epochs * 100,
            batch_size=finetune_batch_size,
            lr=finetune_lr,
            device=device,
        )
        lm_head_path = save_path / "lm_head_finetuned.pt"
        torch.save(model.lm_head.state_dict(), lm_head_path)
        if not quiet:
            print(f"lm_head saved to: {lm_head_path}")
    return rs_model, model, tokenizer


In [ ]:
rs_model, base_model, tokenizer = run_rsulf_colab(
    model_name="mistralai/Mistral-7B-v0.1",
    save_dir="/content/mistral-7b-rsulf",
    folding_ratio=0.5,
    fast_mode=True,
    max_layers=4,
    finetune_text=None,
)
